# Prova pilot: certificats energètics × resposta a la calor (v0.2, 20/10/2026)

**Pregunta de la prova (1 setmana):** les dades obertes de certificats energètics (CEE) de Barcelona tenen prou informació per connectar els paràmetres *declarats* dels edificis amb la resposta *mesurada* a la calor (χ per codi postal, notebook v0.10/v0.11)?

**Què NO és:** un resultat. Amb ~40 codis postals la potència estadística és baixa. Busquem: (1) quins camps hi ha, (2) si es poden agregar per CP, (3) si hi ha senyal que valgui la pena perseguir.

**Canvis v0.2:** (a) corregit l'error del pas 3 (la columna `georeferencia` venia com a diccionari); (b) camps fixats per al registre `j6ii-t3w2`; (c) **no fem servir `energia_refrigeraci` com a "té aire condicionat"**: en la metodologia dels certificats, si l'habitatge no té refrigeració s'hi pot assignar un sistema per defecte, de manera que el valor no indica presència d'equip (cal verificar-ho). En lloc d'això, fem servir la **demanda de refrigeració** (propietat física de l'envolupant) i, a la Part 2, el conjunt d'**instal·lacions**, que sí descriu els equips.

**Passos:** 1 trobar els conjunts de dades · 2 baixar Barcelona (amb cau al Drive) · 3 inventari de camps · 4 neteja · 5 agregació per CP · 6 creuament amb χ · 7 resum per a la reunió.

In [ ]:
# 0. Paràmetres i Drive
from google.colab import drive
drive.mount("/content/drive")
import os, json, time, requests, numpy as np, pandas as pd
import matplotlib.pyplot as plt

CARPETA_LOCAL = "/content/drive/MyDrive/dades_tesi"     # la mateixa carpeta que el notebook v0.11
SORTIDA = f"{CARPETA_LOCAL}/pilot_cee"
os.makedirs(SORTIDA, exist_ok=True)
DOMINI = "analisi.transparenciacatalunya.cat"           # portal de dades obertes de la Generalitat (Socrata)
FITXER_CHI = f"{CARPETA_LOCAL}/resposta_calor_per_cp_v10.csv"   # sortida del v0.10 (canvia-ho si el tens amb un altre nom)
FITXER_RENDA = f"{CARPETA_LOCAL}/renda_cp.csv"
print("Fitxer χ trobat:", os.path.exists(FITXER_CHI), "| renda:", os.path.exists(FITXER_RENDA))

## 1. Trobar els conjunts de dades de certificats al portal
No donem per fet cap identificador: els busquem al catàleg i en llistem les columnes.

In [ ]:
def cataleg(q):
    """Busca conjunts de dades al catàleg Socrata del portal."""
    for base in [f"https://{DOMINI}/api/catalog/v1", "https://api.us.socrata.com/api/catalog/v1"]:
        try:
            r = requests.get(base, params={"q": q, "domains": DOMINI, "limit": 30}, timeout=60)
            r.raise_for_status()
            return r.json().get("results", [])
        except Exception as e:
            print("  catàleg no disponible a", base, "->", e)
    return []

trobats = {}
for q in ["certificats eficiència energètica", "certificació energètica edificis"]:
    for res in cataleg(q):
        rs = res["resource"]
        trobats[rs["id"]] = rs
print(f"{len(trobats)} conjunts trobats:\n")
for i, rs in trobats.items():
    print(f"{i} | {rs['name']} | {len(rs.get('columns_field_name', []))} columnes | actualitzat {rs.get('data_updated_at','?')[:10]}")
# Identificador conegut (verificat a datos.gob.es el 20/10): 725p-iafq = "cerramientos" (envolupant)

In [ ]:
# Tria del conjunt principal: el que té més columnes i el nom conté "certificat" (revisa-ho a mà!)
candidats = {i: rs for i, rs in trobats.items() if "certific" in rs["name"].lower()}
for i, rs in candidats.items():
    print(f"\n=== {i}: {rs['name']}")
    print(rs.get("columns_field_name", []))
ID_PRINCIPAL = max(candidats, key=lambda i: len(candidats[i].get("columns_field_name", []))) if candidats else None
print("\n>>> ID_PRINCIPAL proposat:", ID_PRINCIPAL, "  (si no és el registre general de certificats, escriu-lo a mà aquí)")
# ID_PRINCIPAL = "xxxx-xxxx"

## 2. Baixar només Barcelona (amb cau al Drive)
Busquem automàticament la columna del municipi i baixem per pàgines. La primera vegada pot tardar uns minuts; després es llegeix del Drive.

In [ ]:
def columnes(id_):
    r = requests.get(f"https://{DOMINI}/api/views/{id_}.json", timeout=60); r.raise_for_status()
    return [c["fieldName"] for c in r.json()["columns"]]

def col_municipi(cols):
    for clau in ["poblaci", "municipi", "localitat"]:
        for c in cols:
            if clau in c.lower() and "codi" not in c.lower():
                return c
    raise ValueError(f"No trobo la columna del municipi entre: {cols}")

def baixa_barcelona(id_, pagina=50000):
    cols = columnes(id_); cm = col_municipi(cols)
    print(f"  columna del municipi: {cm}")
    trossos, offset = [], 0
    while True:
        params = {"$where": f"upper({cm})='BARCELONA'", "$limit": pagina, "$offset": offset, "$order": ":id"}
        for intent in range(4):
            try:
                r = requests.get(f"https://{DOMINI}/resource/{id_}.json", params=params, timeout=300)
                r.raise_for_status(); break
            except Exception as e:
                print(f"  error ({e}); espero 30 s"); time.sleep(30)
        else:
            raise RuntimeError("El portal no respon; torna-ho a provar més tard")
        bloc = pd.DataFrame(r.json())
        if bloc.empty: break
        trossos.append(bloc); offset += pagina
        print(f"  {offset:,} files demanades…")
    return pd.concat(trossos, ignore_index=True)

ruta_cee = f"{SORTIDA}/cee_barcelona_{ID_PRINCIPAL}.csv"
if os.path.exists(ruta_cee):
    cee = pd.read_csv(ruta_cee, dtype=str); print(f"[cau] llegit del Drive: {len(cee):,} certificats")
else:
    cee = baixa_barcelona(ID_PRINCIPAL); cee.to_csv(ruta_cee, index=False)
    print(f"[cau] baixat i desat: {len(cee):,} certificats")
    cee = pd.read_csv(ruta_cee, dtype=str)   # rellegim del CSV: tot com a text (evita columnes amb diccionaris)

## 3. Inventari de camps
**Aquesta és la sortida més important de la setmana.** Fes-ne captura sencera.

In [ ]:
inv = pd.DataFrame({
    "% omplert": cee.apply(lambda s: (s.notna() & ~s.astype(str).str.strip().isin(["", "nan", "None"])).mean() * 100).round(1),
    "valors diferents": cee.nunique(),
    "exemple": cee.apply(lambda s: s.dropna().astype(str).iloc[0][:40] if s.notna().any() else ""),
})
pd.set_option("display.max_rows", 300)
display(inv)
inv.to_csv(f"{SORTIDA}/inventari_camps.csv")

In [ ]:
# Camps del registre j6ii-t3w2 (noms verificats al teu catàleg del 20/10)
F = {
    "cp": "codi_postal", "cadastre": "referencia_cadastral", "any": "any_construccio",
    "us": "us_edifici", "qual_ep": "qualificaci_de_consum_d", "ep_m2": "energia_prim_ria_no_renovable",
    "co2_m2": "emissions_de_co2", "superficie": "metres_cadastre", "data": "data_entrada",
    "lat": "latitud", "lon": "longitud",
    # físics (kWh/m² any, segons el certificat)
    "dem_refr": "energia_refrigeraci_demanda", "dem_calef": "energia_calefacci_demanda",
    "en_refr": "energia_refrigeraci",
    # envolupant (valors declarats; caldrà mirar-ne les unitats)
    "aillaments": "valor_aillaments", "finestres": "valor_finestres",
    # rehabilitació i inspecció tècnica
    "rehab": "rehabilitacio_energetica", "ite": "informe_ins_tecnica_edifici",
}
falten = [k for k, c in F.items() if c not in cee.columns]
print("Camps que no hi són:", falten)
for k in falten: F[k] = None
# Quina és la qualificació global? Mirem totes les columnes de qualificació
for c in [c for c in cee.columns if c.startswith("qualificaci")]:
    print(f"\n--- {c} ---"); print(cee[c].value_counts(dropna=False).head(9).to_dict())
for k in ["us", "rehab", "ite"]:
    if F[k]: print(f"\n--- {k} ({F[k]}) ---"); print(cee[F[k]].value_counts(dropna=False).head(12))
print("\nResum numèric dels camps físics (tots els certificats de Barcelona):")
num = lambda s: pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")
display(pd.DataFrame({k: num(cee[F[k]]) for k in ["ep_m2", "dem_refr", "dem_calef", "en_refr", "aillaments", "finestres"] if F[k]}).describe().round(2))
print("% certificats amb energia_refrigeraci > 0:", round(100 * (num(cee[F["en_refr"]]) > 0).mean(), 1),
      " <- si surt prop del 100 %, és un indici que inclou sistemes per defecte (no presència d'equip)")

## 4. Neteja
- Només ús residencial.
- Un certificat per referència cadastral (el més recent): un pis pot tenir-ne diversos.
- **Biaix de selecció:** els certificats només existeixen quan es ven o es lloga (o per ajuts). No són una mostra aleatòria del parc. Ho mesurem amb la cobertura per CP.

In [ ]:
d = cee.copy()
num = lambda s: pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")
d["cp"] = d[F["cp"]].astype(str).str.extract(r"(\d{5})")[0] if F["cp"] else np.nan
d = d[d["cp"].str.startswith("080", na=False)]
if F["us"]:
    res = d[F["us"]].astype(str).str.lower().str.contains("habitatge|residencial|vivienda|bloc", na=False)
    print(f"Residencial: {res.sum():,} de {len(d):,}"); d = d[res]
if F["data"]:
    d["data_cee"] = pd.to_datetime(d[F["data"]], errors="coerce")
if F["cadastre"]:
    n0 = len(d)
    d = d.sort_values("data_cee" if "data_cee" in d else F["cadastre"]).drop_duplicates(F["cadastre"], keep="last")
    print(f"Duplicats per referència cadastral eliminats: {n0 - len(d):,}")
d["any"]  = num(d[F["any"]]) if F["any"] else np.nan
d["ep"]   = num(d[F["ep_m2"]]) if F["ep_m2"] else np.nan
d["co2"]  = num(d[F["co2_m2"]]) if F["co2_m2"] else np.nan
d["sup"]  = num(d[F["superficie"]]) if F["superficie"] else np.nan
d["lletra"] = d[F["qual_ep"]].astype(str).str.upper().str.extract(r"\b([A-G])\b")[0] if F["qual_ep"] else np.nan
for k in ["dem_refr", "dem_calef", "en_refr", "aillaments", "finestres"]:
    d[k] = num(d[F[k]]) if F[k] else np.nan
    d.loc[(d[k] < 0) | (d[k] > 1000), k] = np.nan
si_ = lambda c: d[c].astype(str).str.strip().str.lower().isin(["sí", "si", "s", "true", "1", "1.0", "yes"]).astype(float)
d["rehab"] = si_(F["rehab"]) if F["rehab"] else np.nan
d["ite"]   = si_(F["ite"]) if F["ite"] else np.nan
d.loc[(d["any"] < 1800) | (d["any"] > 2026), "any"] = np.nan
d.loc[(d["ep"] <= 0) | (d["ep"] > 2000), "ep"] = np.nan
print(f"Certificats residencials nets: {len(d):,}  |  CP diferents: {d['cp'].nunique()}")
print(d[["any", "ep", "co2", "sup", "dem_refr", "dem_calef", "aillaments", "finestres", "rehab", "ite"]].describe().round(1))

## 5. Agregació per codi postal

In [ ]:
g = d.groupby("cp")
cp_cee = pd.DataFrame({
    "n_cee":        g.size(),
    "any_mediana":  g["any"].median(),
    "pct_pre1980":  g["any"].apply(lambda s: 100 * (s < 1980).mean()),
    "ep_mediana":   g["ep"].median(),
    "co2_mediana":  g["co2"].median(),
    "pct_FG":       g["lletra"].apply(lambda s: 100 * s.isin(["F", "G"]).mean()),
    "pct_AB":       g["lletra"].apply(lambda s: 100 * s.isin(["A", "B"]).mean()),
    "dem_refr_med": g["dem_refr"].median(),
    "dem_calef_med": g["dem_calef"].median(),
    "aillaments_med": g["aillaments"].median(),
    "finestres_med": g["finestres"].median(),
    "pct_rehab":    100 * g["rehab"].mean(),
    "pct_ite":      100 * g["ite"].mean(),
}).round(2)
display(cp_cee.sort_values("n_cee"))
cp_cee.to_csv(f"{SORTIDA}/cee_per_cp.csv")
print("CP amb menys de 200 certificats (poc fiables):", list(cp_cee.index[cp_cee.n_cee < 200]))

## 6. Creuament amb la resposta a la calor
Fem servir el χ residencial per CP del v0.10 (`chi_estatica`, efecte d'1 grau-dia de refredament en el consum, en tant per u) i la **forma de la memòria**: la fracció de la resposta que arriba amb retard (`chi_lag1+2+3` sobre el total), que hauria de créixer amb la massa tèrmica (edificis antics de parets gruixudes).

Correlacions de Spearman, i també **parcials controlant per la renda** (la renda explica molt: pisos rics, més aire condicionat). Amb ~40 punts, |ρ| < 0,32 no és significatiu (p > 0,05).

In [ ]:
chi = pd.read_csv(FITXER_CHI, dtype={"cp": str})
chi = chi[chi["sector"].str.lower().str.startswith("resid")].set_index("cp")
lags = chi[["chi_lag0", "chi_lag1", "chi_lag2", "chi_lag3"]]
chi["frac_retard"] = lags[["chi_lag1", "chi_lag2", "chi_lag3"]].sum(axis=1) / lags.sum(axis=1)
T = cp_cee.join(chi[["chi_estatica", "chi_estatica_se", "frac_retard"]], how="inner")
if os.path.exists(FITXER_RENDA):
    T = T.join(pd.read_csv(FITXER_RENDA, dtype={"cp": str}).set_index("cp")["renda"])
T = T[T.n_cee >= 200]
print(f"CP al creuament: {len(T)}")

from scipy import stats
def parcial(x, y, z):
    """Spearman parcial: correlació dels residus dels rangs de x i y després de treure z."""
    r = lambda v: stats.rankdata(v)
    X, Y, Z = r(x), r(y), r(z)
    rx = X - np.polyval(np.polyfit(Z, X, 1), Z); ry = Y - np.polyval(np.polyfit(Z, Y, 1), Z)
    return stats.pearsonr(rx, ry)
files = []
for obs in ["chi_estatica", "frac_retard"]:
    for var in ["any_mediana", "pct_pre1980", "ep_mediana", "pct_FG", "pct_AB", "dem_refr_med", "aillaments_med", "finestres_med", "pct_rehab", "renda"]:
        if var not in T or T[var].isna().all(): continue
        m = T[[obs, var] + (["renda"] if "renda" in T and var != "renda" else [])].dropna()
        rho, p = stats.spearmanr(m[obs], m[var])
        rp, pp = parcial(m[obs], m[var], m["renda"]) if "renda" in m and var != "renda" else (np.nan, np.nan)
        files.append(dict(observable=obs, variable_cee=var, n=len(m), rho=round(rho, 2), p=round(p, 3),
                          rho_parcial_renda=round(rp, 2), p_parcial=round(pp, 3)))
corr = pd.DataFrame(files); display(corr)
corr.to_csv(f"{SORTIDA}/correlacions_pilot.csv", index=False)

In [ ]:
# Figures: les dues relacions més informatives
pars = [("dem_refr_med", "chi_estatica", "Demanda de refrigeració declarada (kWh/m² any, mediana)", "χ residencial (per grau-dia)"),
        ("any_mediana", "frac_retard", "Any de construcció (mediana)", "Fracció de resposta retardada"),
        ("ep_mediana", "chi_estatica", "Energia primària declarada (kWh/m² any)", "χ residencial (per grau-dia)")]
pars = [p for p in pars if p[0] in T and T[p[0]].notna().any()]
fig, ax = plt.subplots(1, len(pars), figsize=(5 * len(pars), 4))
ax = np.atleast_1d(ax)
for a, (x, y, xl, yl) in zip(ax, pars):
    a.scatter(T[x], T[y], c=T.get("renda"), cmap="viridis", s=30)
    for cp, r in T.iterrows(): a.annotate(cp[-2:], (r[x], r[y]), fontsize=7, alpha=.6)
    a.set_xlabel(xl); a.set_ylabel(yl)
fig.suptitle("Pilot: paràmetres declarats (certificats) vs resposta mesurada (Datadis). Color = renda")
fig.tight_layout(); fig.savefig(f"{SORTIDA}/fig_pilot.png", dpi=150); plt.show()

## Part 2 (opcional si dona temps): els equips reals, del conjunt d'instal·lacions
El conjunt `cmnw-duhh` descriu els sistemes de cada certificat (tipus de generador, potència de fred…). Si hi ha equips de fred declarats, podem calcular **quina fracció d'habitatges té aire condicionat per CP** i provar si explica el creixement de χ (factor λ). Primer només mirem què hi ha.

In [ ]:
ID_INST = "cmnw-duhh"
ruta_inst = f"{SORTIDA}/instal_barcelona.csv"
if os.path.exists(ruta_inst):
    inst = pd.read_csv(ruta_inst, dtype=str); print(f"[cau] instal·lacions llegides: {len(inst):,} files")
else:
    inst = baixa_barcelona(ID_INST); inst.to_csv(ruta_inst, index=False)
    inst = pd.read_csv(ruta_inst, dtype=str); print(f"[cau] instal·lacions baixades: {len(inst):,} files")
print("Certificats diferents:", inst["num_cas"].nunique())
for c in ["us_edifici", "it_gen_tipus", "it_gen_categoria", "it_gen_vector_energetico", "it_ss_tipus", "it_alt_tipus"]:
    if c in inst: print(f"\n--- {c} ---"); print(inst[c].value_counts(dropna=False).head(15))
num = lambda s: pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")
print("\nFiles amb potència de fred > 0:", int((num(inst.get("it_ss_potencia_frio", pd.Series(dtype=str))) > 0).sum()))

In [ ]:
# Fracció de certificats residencials amb equip de fred declarat, per CP (provisional: revisa les categories de dalt)
inst["fred"] = (num(inst["it_ss_potencia_frio"]) > 0) | inst["it_gen_tipus"].astype(str).str.lower().str.contains("bomba|refriger|enfri|frío|fred", na=False)
per_cas = inst.groupby("num_cas")["fred"].any()
dd = d[["num_cas", "cp"]].drop_duplicates("num_cas").set_index("num_cas").join(per_cas, how="inner")
print(f"Certificats residencials nets amb dades d'instal·lacions: {len(dd):,} de {len(d):,}")
fred_cp = dd.groupby("cp")["fred"].agg(["mean", "size"]).rename(columns={"mean": "frac_fred", "size": "n_inst"})
fred_cp["frac_fred"] *= 100
T2 = T.join(fred_cp, how="left")
display(T2[["chi_estatica", "frac_fred", "n_inst", "renda"]].sort_values("frac_fred").round(2))
m = T2[["chi_estatica", "frac_fred", "renda"]].dropna()
if len(m) > 10:
    print("Spearman χ vs % amb fred:", stats.spearmanr(m.chi_estatica, m.frac_fred))
    print("Parcial controlant renda:", parcial(m.chi_estatica, m.frac_fred, m.renda))
fred_cp.to_csv(f"{SORTIDA}/fred_per_cp.csv")

## 7. Resum per a la reunió
Genera un text curt amb el que hem trobat. **Envia'm aquest text, l'inventari (pas 3) i la figura.**

In [ ]:
def si(x): return "sí" if x else "NO"
txt = f"""PILOT CEE × CALOR ({pd.Timestamp.today():%d/%m/%Y})
Conjunt: {ID_PRINCIPAL} | certificats Barcelona bruts: {len(cee):,} | residencials nets: {len(d):,} | CP: {d['cp'].nunique()}
% amb energia_refrigeraci>0: {100*(d['en_refr']>0).mean():.1f} | % rehabilitació energètica: {100*d['rehab'].mean():.1f} | % amb ITE: {100*d['ite'].mean():.1f}
Cobertura mediana: {cp_cee.n_cee.median():.0f} certificats per CP (mínim {cp_cee.n_cee.min()})
Correlacions (Spearman, n≈{len(T)}):
{corr.to_string(index=False)}
"""
print(txt)
open(f"{SORTIDA}/resum_pilot.txt", "w").write(txt)
import shutil; shutil.make_archive("/content/pilot_cee_per_claude", "zip", SORTIDA)
from google.colab import files as _f; _f.download("/content/pilot_cee_per_claude.zip")